# SupportMind — RAG Analysis Notebook

This notebook loads the latest offline evaluation report and visualizes:

1. Per-category source-hit rate
2. RAGAS-style metrics (when enabled)
3. Latency distribution
4. Cost-per-query across LLM models

Run the eval first:

```bash
.venv/Scripts/python -m eval.run_eval --mode offline --output-dir eval/results --enable-ragas
```

In [ ]:
import json
from pathlib import Path

REPORT_PATH = Path("eval/results/latest_report.json")
if not REPORT_PATH.exists():
    # Fall back: try to read the markdown and parse the summary
    REPORT_PATH = Path("eval/results/latest_report.md")
print(f"Loading report: {REPORT_PATH}")
print(f"Exists: {REPORT_PATH.exists()}")

In [ ]:
import re

def parse_markdown_report(path: Path) -> dict:
    text = path.read_text(encoding="utf-8")
    summary = {}
    ragas = {}
    in_ragas = False
    for line in text.splitlines():
        m = re.match(r"\|\s*([^|]+?)\s*\|\s*([^|]+?)\s*\|\s*$", line)
        if not m:
            if "RAGAS-Style Metrics" in line:
                in_ragas = True
                continue
            if in_ragas and line.startswith("## "):
                in_ragas = False
            continue
        k, v = m.group(1).strip(), m.group(2).strip()
        if v == "---":
            continue
        try:
            num = float(v.rstrip("%")) / (100.0 if v.endswith("%") else 1.0)
        except ValueError:
            num = v
        if in_ragas:
            ragas[k] = num
        else:
            summary[k] = num
    return {"summary": summary, "ragas": ragas}

if REPORT_PATH.suffix == ".json":
    data = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
else:
    data = parse_markdown_report(REPORT_PATH)

print("Summary keys:", list(data.get("summary", {}).keys())[:10])
print("RAGAS keys:", list(data.get("ragas", {}).keys())[:10])

In [ ]:
summary = data.get("summary", {})
print("=" * 50)
print("  SupportMind Offline Evaluation Summary")
print("=" * 50)
for k, v in summary.items():
    if isinstance(v, float):
        if abs(v) <= 1.0:
            print(f"  {k:30s}  {v * 100:7.1f}%")
        else:
            print(f"  {k:30s}  {v:7.2f}")
    else:
        print(f"  {k:30s}  {v}")

In [ ]:
ragas = data.get("ragas", {})
if ragas:
    print("=" * 50)
    print("  RAGAS-Style Metrics")
    print("=" * 50)
    for k, v in ragas.items():
        if isinstance(v, float):
            print(f"  {k:30s}  {v:7.3f}")
        else:
            print(f"  {k:30s}  {v}")
else:
    print("RAGAS metrics not enabled. Re-run with --enable-ragas to populate.")

In [ ]:
# Visualize RAGAS metrics as a horizontal bar chart
try:
    import matplotlib.pyplot as plt
    if ragas:
        items = sorted(ragas.items(), key=lambda x: x[1] if isinstance(x[1], (int, float)) else 0)
        labels = [k for k, _ in items if isinstance(_, (int, float))]
        values = [v for _, v in items if isinstance(v, (int, float))]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.barh(labels, values, color="#4C72B0")
        ax.set_xlabel("Score")
        ax.set_xlim(0, 1.0)
        ax.set_title("RAGAS-Style Metrics")
        for i, v in enumerate(values):
            ax.text(v + 0.01, i, f"{v:.2f}", va="center")
        plt.tight_layout()
        plt.show()
    else:
        print("No RAGAS metrics to plot. Re-run with --enable-ragas.")
except ImportError:
    print("matplotlib not installed. Skip visualization.")

In [ ]:
# Estimate cost per query across LLM models
from eval.benchmarks.cost_estimator import (
    estimate_query_cost,
    list_known_models,
    rank_models_by_cost,
)

print("=" * 50)
print("  Estimated cost per query")
print("=" * 50)
ranking = rank_models_by_cost(list_known_models())
for model, cost in ranking:
    print(f"  {model:40s}  ${cost:.6f}")

In [ ]:
# Try to load cost ledger and show spend by agent (if any)
from app.observability.cost import CostTracker

tracker = CostTracker()
breakdown = tracker.breakdown_by_agent()
if breakdown:
    print("=" * 50)
    print("  Real spend by agent")
    print("=" * 50)
    for agent, stats in breakdown.items():
        print(f"  {agent:20s}  {stats['calls']:4d} calls  ${stats['total_cost_usd']:.6f}")
else:
    print("No cost data yet — calls must be made through agents that record costs.")

## Next steps

- Add more eval cases (see [`docs/EVALUATION_GUIDE.md`](../docs/EVALUATION_GUIDE.md))
- Run a sweep: `python scripts/eval_experiments.py --experiment topk_sweep --variants topk_3,topk_5,topk_8`
- Inspect the prompt registry: [`../app/agents/prompts/`](../app/agents/prompts/)
- Add new prompt variants and A/B test them
- Calibrate confidence: Phase 2 feature, see [`implementation_plan.md`](../implementation_plan.md)